### Step 1: Read the Raw Files into a DataFrame

In [0]:
# STEP 1: Stage raw binary streams from the volume
VOLUME_PATH = "/Volumes/main/default/enforcement_tracker"

raw_binary_df = (spark.read
                 .format("binaryFile")
                 .option("pathGlobFilter", "*.pdf")
                 .load(VOLUME_PATH))

# Let's inspect the staged schema and metadata
print(f"Discovered {raw_binary_df.count()} files in the volume.")
display(raw_binary_df.select("path", "modificationTime", "length"))

### Step 2: Run the AI Parser to Generate the Raw VARIANT Data

In [0]:
# STEP 2: Pass binary payloads to the AI Model
from pyspark.sql.functions import expr, col

parsed_raw_df = raw_binary_df.withColumn("ai_output", expr("ai_parse_document(content)"))

# Display the source path alongside the massive raw semi-structured AI object
display(parsed_raw_df.select(
    expr("element_at(split(path, '/'), -1)").alias("file_name"), 
    col("ai_output")
))

### Step 3: Inspect the Isolated Document Elements (JSON Schema View)

In [0]:
# STEP 3: Unpack the structural element array schema (FIXED)
from pyspark.sql.functions import expr, col

elements_preview_df = parsed_raw_df.select(
    expr("element_at(split(path, '/'), -1)").alias("file_name"),
    expr("ai_output:metadata.page_count").cast("int").alias("pages"),
    # Fix: We explicitly cast the variant path to a string so from_json can parse it safely
    expr("""
        from_json(
            cast(ai_output:document.elements as string), 
            'array<struct<type:string,content:string>>'
        )
    """).alias("layout_elements")
)

display(elements_preview_df)

### Step 4: Flatten the Blocks into Clean Reading-Order Text

In [0]:
# STEP 4: Bulletproof array extraction for page boundaries
from pyspark.sql.functions import expr

final_flattened_df = parsed_raw_df.select(
    expr("element_at(split(path, '/'), -1)").alias("source_file_name"),
    
    # FIX: Isolate page numbers as an integer array, then pull the maximum value natively
    expr("""
        coalesce(
            cast(ai_output:document.metadata.page_count as int),
            array_max(
                transform(
                    from_json(cast(ai_output:document.elements as string), 'array<struct<page_number:int>>'),
                    el -> cast(el.page_number as int)
                )
            ),
            1
        )
    """).alias("total_pages"),
    
    expr("""
        concat_ws('\\n\\n', 
            transform(
                from_json(
                    cast(ai_output:document.elements as string), 
                    'array<struct<type:string,content:string>>'
                ),
                element -> element.content
            )
        )
    """).alias("raw_text_payload")
)

display(final_flattened_df)

### Step 5: Save the Safe Matrix to Your Unity Catalog Table
Write as persistent permanent Delta table asset inside database.

In [0]:
# STEP 5: Commit the verified records to permanent Delta storage
TABLE_TARGET = "main.default.gdpr_raw_multilingual_precedents"

(final_flattened_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_TARGET))

# Turn on Change Data Feed for version history tracking
spark.sql(f"ALTER TABLE {TABLE_TARGET} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"🎉 Table successfully compiled and locked down at: {TABLE_TARGET}")

### Step 6: Create the Translation Enrichment Cell

In [0]:
%sql
# STEP 6: BATCH PROCESSING - Chunking + Translation with Progress Tracking
from pyspark.sql.functions import col, expr, explode, concat_ws, struct, array_sort, collect_list, lit
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType
import re

# ============================================================================
# CONFIGURATION
# ============================================================================
BATCH_SIZE = 50  # Process 50 documents at a time (~23 minutes per batch)
MAX_TOKENS = 1200  # Proven safe from testing

# ============================================================================
# CHECK WHAT'S ALREADY PROCESSED
# ============================================================================
print("Checking for existing progress...")
try:
    processed_df = spark.table("main.default.gdpr_translated_chunks")
    already_processed = [row.source_file_name for row in processed_df.select("source_file_name").distinct().collect()]
    print(f"✓ Found {len(already_processed)} already-processed documents")
except:
    already_processed = []
    print("✓ No previous progress found - starting fresh")

# ============================================================================
# LOAD UNPROCESSED DOCUMENTS
# ============================================================================
print("\nLoading raw documents...")
all_docs_df = spark.table("main.default.gdpr_raw_multilingual_precedents")
total_docs = all_docs_df.count()

# Filter out already-processed documents
if already_processed:
    unprocessed_df = all_docs_df.filter(~col("source_file_name").isin(already_processed))
else:
    unprocessed_df = all_docs_df

remaining_docs = unprocessed_df.count()

print(f"Total documents: {total_docs}")
print(f"Already processed: {len(already_processed)}")
print(f"Remaining: {remaining_docs}")

if remaining_docs == 0:
    print("\n🎉 All documents already processed!")
    dbutils.notebook.exit("Complete")

# ============================================================================
# TAKE NEXT BATCH
# ============================================================================
batch_df = unprocessed_df.limit(BATCH_SIZE)
batch_count = batch_df.count()

print(f"\n{'='*70}")
print(f"PROCESSING BATCH: {batch_count} documents")
print(f"Remaining after this batch: {remaining_docs - batch_count}")
print(f"{'='*70}")

# ============================================================================
# CHUNKING FUNCTION
# ============================================================================
def estimate_tokens(text):
    if not text:
        return 0
    words = len(text.split())
    return int(words * 1.25)

def paragraph_based_chunking(text):
    if not text:
        return []
    
    if estimate_tokens(text) <= MAX_TOKENS:
        return [{"chunk_index": 0, "chunk_text": text, "chunk_tokens": estimate_tokens(text)}]
    
    paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    chunk_index = 0
    
    for para in paragraphs:
        para_tokens = estimate_tokens(para)
        
        if para_tokens > MAX_TOKENS:
            if current_chunk:
                chunks.append({
                    "chunk_index": chunk_index,
                    "chunk_text": "\n\n".join(current_chunk),
                    "chunk_tokens": current_tokens
                })
                chunk_index += 1
                current_chunk = []
                current_tokens = 0
            
            lines = [line.strip() for line in para.split('\n') if line.strip()]
            for line in lines:
                line_tokens = estimate_tokens(line)
                if current_tokens + line_tokens <= MAX_TOKENS:
                    current_chunk.append(line)
                    current_tokens += line_tokens
                else:
                    if current_chunk:
                        chunks.append({
                            "chunk_index": chunk_index,
                            "chunk_text": "\n".join(current_chunk),
                            "chunk_tokens": current_tokens
                        })
                        chunk_index += 1
                    current_chunk = [line]
                    current_tokens = line_tokens
            continue
        
        if current_tokens + para_tokens <= MAX_TOKENS:
            current_chunk.append(para)
            current_tokens += para_tokens
        else:
            if current_chunk:
                chunks.append({
                    "chunk_index": chunk_index,
                    "chunk_text": "\n\n".join(current_chunk),
                    "chunk_tokens": current_tokens
                })
                chunk_index += 1
            current_chunk = [para]
            current_tokens = para_tokens
    
    if current_chunk:
        chunks.append({
            "chunk_index": chunk_index,
            "chunk_text": "\n\n".join(current_chunk),
            "chunk_tokens": current_tokens
        })
    
    return chunks

# Register UDF
chunk_schema = ArrayType(StructType([
    StructField("chunk_index", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("chunk_tokens", IntegerType(), False)
]))

spark.udf.register("paragraph_chunk_udf", paragraph_based_chunking, chunk_schema)

# ============================================================================
# CHUNK THE BATCH
# ============================================================================
print("\nChunking documents...")
chunked_df = batch_df.withColumn(
    "chunks",
    expr("paragraph_chunk_udf(raw_text_payload)")
).withColumn(
    "chunk_exploded",
    explode(col("chunks"))
).select(
    col("source_file_name"),
    col("total_pages"),
    col("chunk_exploded.chunk_index").alias("chunk_index"),
    col("chunk_exploded.chunk_text").alias("chunk_text"),
    col("chunk_exploded.chunk_tokens").alias("chunk_tokens")
)

total_chunks = chunked_df.count()
print(f"✓ Created {total_chunks} chunks from {batch_count} documents")
print(f"⏱️  Estimated translation time: {total_chunks * 20 // 60} minutes")

# ============================================================================
# TRANSLATE THE BATCH
# ============================================================================
print(f"\n{'='*70}")
print("TRANSLATING CHUNKS")
print(f"{'='*70}")

translated_df = chunked_df.withColumn(
    "translated_text",
    expr("ai_translate(chunk_text, 'en')")
)

# ============================================================================
# SAVE PROGRESS (APPEND MODE)
# ============================================================================
print("\nSaving progress...")
if already_processed:
    # Append to existing table
    (translated_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("main.default.gdpr_translated_chunks"))
else:
    # Create new table
    (translated_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("main.default.gdpr_translated_chunks"))

print("✓ Batch saved!")

# ============================================================================
# PROGRESS SUMMARY
# ============================================================================
total_processed = spark.table("main.default.gdpr_translated_chunks") \
    .select("source_file_name").distinct().count()

print(f"\n{'='*70}")
print("BATCH COMPLETE")
print(f"{'='*70}")
print(f"Documents processed this batch: {batch_count}")
print(f"Total documents processed: {total_processed}/{total_docs}")
print(f"Remaining: {total_docs - total_processed}")
print(f"\n👉 RE-RUN THIS CELL to process the next batch")
print(f"{'='*70}")

In [0]:
%sql
# STEP 6: BATCH PROCESSING - Chunking + Translation with Progress Tracking
from pyspark.sql.functions import col, expr, explode, concat_ws, struct, array_sort, collect_list, lit
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType
import re

# ============================================================================
# CONFIGURATION
# ============================================================================
BATCH_SIZE = 50  # Process 50 documents at a time (~23 minutes per batch)
MAX_TOKENS = 1200  # Proven safe from testing

# ============================================================================
# CHECK WHAT'S ALREADY PROCESSED
# ============================================================================
print("Checking for existing progress...")
try:
    processed_df = spark.table("main.default.gdpr_translated_chunks")
    already_processed = [row.source_file_name for row in processed_df.select("source_file_name").distinct().collect()]
    print(f"✓ Found {len(already_processed)} already-processed documents")
except:
    already_processed = []
    print("✓ No previous progress found - starting fresh")

# ============================================================================
# LOAD UNPROCESSED DOCUMENTS
# ============================================================================
print("\nLoading raw documents...")
all_docs_df = spark.table("main.default.gdpr_raw_multilingual_precedents")
total_docs = all_docs_df.count()

# Filter out already-processed documents
if already_processed:
    unprocessed_df = all_docs_df.filter(~col("source_file_name").isin(already_processed))
else:
    unprocessed_df = all_docs_df

remaining_docs = unprocessed_df.count()

print(f"Total documents: {total_docs}")
print(f"Already processed: {len(already_processed)}")
print(f"Remaining: {remaining_docs}")

if remaining_docs == 0:
    print("\n🎉 All documents already processed!")
    dbutils.notebook.exit("Complete")

# ============================================================================
# TAKE NEXT BATCH
# ============================================================================
batch_df = unprocessed_df.limit(BATCH_SIZE)
batch_count = batch_df.count()

print(f"\n{'='*70}")
print(f"PROCESSING BATCH: {batch_count} documents")
print(f"Remaining after this batch: {remaining_docs - batch_count}")
print(f"{'='*70}")

# ============================================================================
# CHUNKING FUNCTION
# ============================================================================
def estimate_tokens(text):
    if not text:
        return 0
    words = len(text.split())
    return int(words * 1.25)

def paragraph_based_chunking(text):
    if not text:
        return []
    
    if estimate_tokens(text) <= MAX_TOKENS:
        return [{"chunk_index": 0, "chunk_text": text, "chunk_tokens": estimate_tokens(text)}]
    
    paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    chunk_index = 0
    
    for para in paragraphs:
        para_tokens = estimate_tokens(para)
        
        if para_tokens > MAX_TOKENS:
            if current_chunk:
                chunks.append({
                    "chunk_index": chunk_index,
                    "chunk_text": "\n\n".join(current_chunk),
                    "chunk_tokens": current_tokens
                })
                chunk_index += 1
                current_chunk = []
                current_tokens = 0
            
            lines = [line.strip() for line in para.split('\n') if line.strip()]
            for line in lines:
                line_tokens = estimate_tokens(line)
                if current_tokens + line_tokens <= MAX_TOKENS:
                    current_chunk.append(line)
                    current_tokens += line_tokens
                else:
                    if current_chunk:
                        chunks.append({
                            "chunk_index": chunk_index,
                            "chunk_text": "\n".join(current_chunk),
                            "chunk_tokens": current_tokens
                        })
                        chunk_index += 1
                    current_chunk = [line]
                    current_tokens = line_tokens
            continue
        
        if current_tokens + para_tokens <= MAX_TOKENS:
            current_chunk.append(para)
            current_tokens += para_tokens
        else:
            if current_chunk:
                chunks.append({
                    "chunk_index": chunk_index,
                    "chunk_text": "\n\n".join(current_chunk),
                    "chunk_tokens": current_tokens
                })
                chunk_index += 1
            current_chunk = [para]
            current_tokens = para_tokens
    
    if current_chunk:
        chunks.append({
            "chunk_index": chunk_index,
            "chunk_text": "\n\n".join(current_chunk),
            "chunk_tokens": current_tokens
        })
    
    return chunks

# Register UDF
chunk_schema = ArrayType(StructType([
    StructField("chunk_index", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("chunk_tokens", IntegerType(), False)
]))

spark.udf.register("paragraph_chunk_udf", paragraph_based_chunking, chunk_schema)

# ============================================================================
# CHUNK THE BATCH
# ============================================================================
print("\nChunking documents...")
chunked_df = batch_df.withColumn(
    "chunks",
    expr("paragraph_chunk_udf(raw_text_payload)")
).withColumn(
    "chunk_exploded",
    explode(col("chunks"))
).select(
    col("source_file_name"),
    col("total_pages"),
    col("chunk_exploded.chunk_index").alias("chunk_index"),
    col("chunk_exploded.chunk_text").alias("chunk_text"),
    col("chunk_exploded.chunk_tokens").alias("chunk_tokens")
)

total_chunks = chunked_df.count()
print(f"✓ Created {total_chunks} chunks from {batch_count} documents")
print(f"⏱️  Estimated translation time: {total_chunks * 20 // 60} minutes")

# ============================================================================
# TRANSLATE THE BATCH
# ============================================================================
print(f"\n{'='*70}")
print("TRANSLATING CHUNKS")
print(f"{'='*70}")

translated_df = chunked_df.withColumn(
    "translated_text",
    expr("ai_translate(chunk_text, 'en')")
)

# ============================================================================
# SAVE PROGRESS (APPEND MODE)
# ============================================================================
print("\nSaving progress...")
if already_processed:
    # Append to existing table
    (translated_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("main.default.gdpr_translated_chunks"))
else:
    # Create new table
    (translated_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("main.default.gdpr_translated_chunks"))

print("✓ Batch saved!")

# ============================================================================
# PROGRESS SUMMARY
# ============================================================================
total_processed = spark.table("main.default.gdpr_translated_chunks") \
    .select("source_file_name").distinct().count()

print(f"\n{'='*70}")
print("BATCH COMPLETE")
print(f"{'='*70}")
print(f"Documents processed this batch: {batch_count}")
print(f"Total documents processed: {total_processed}/{total_docs}")
print(f"Remaining: {total_docs - total_processed}")
print(f"\n👉 RE-RUN THIS CELL to process the next batch")
print(f"{'='*70}")

In [0]:
# Quick check for translation truncation
from pyspark.sql.functions import expr, col

validation = spark.table("main.default.gdpr_chunks_test_translated") \
    .withColumn("original_len", expr("length(chunk_text)")) \
    .withColumn("translated_len", expr("length(translated_text)")) \
    .withColumn("ratio", expr("translated_len / original_len"))

# Show summary stats
print("📊 Translation Quality Summary:")
validation.selectExpr(
    "count(*) as total_chunks",
    "round(min(ratio), 2) as min_ratio",
    "round(avg(ratio), 2) as avg_ratio", 
    "round(max(ratio), 2) as max_ratio"
).show()

# Check for truncated chunks (ratio < 0.5)
truncated = validation.filter(col("ratio") < 0.5)
truncated_count = truncated.count()

print(f"\n🔍 Truncation Check:")
if truncated_count > 0:
    print(f"⚠️ WARNING: {truncated_count} chunks appear truncated (ratio < 0.5)")
    truncated.select(
        "chunk_index", 
        "chunk_tokens",
        "original_len", 
        "translated_len",
        expr("round(ratio, 2)").alias("ratio")
    ).show()
else:
    print("✓ No truncation detected! All translations look complete.")

# Show chunks with suspicious ratios (< 0.7)
suspicious = validation.filter((col("ratio") < 0.7) & (col("ratio") >= 0.5))
suspicious_count = suspicious.count()
if suspicious_count > 0:
    print(f"\n⚠️ {suspicious_count} chunks have low ratios (0.5-0.7) - may need review:")
    suspicious.select(
        "chunk_index",
        "chunk_tokens", 
        expr("round(ratio, 2)").alias("ratio")
    ).show()

In [0]:
# Manual inspection of chunk endings
from pyspark.sql.functions import col

chunk_0 = spark.table("main.default.gdpr_chunks_test_translated") \
    .filter(col("chunk_index") == 0) \
    .select("chunk_text", "translated_text", "chunk_tokens") \
    .collect()[0]

print("="*70)
print(f"CHUNK 0 INSPECTION (Token count: {chunk_0['chunk_tokens']})")
print("="*70)

print("\n🔍 ORIGINAL - Last 300 chars:")
print("-"*70)
print(chunk_0["chunk_text"][-300:])

print("\n\n🔍 TRANSLATED - Last 300 chars:")
print("-"*70)
print(chunk_0["translated_text"][-300:])

print("\n\n📊 ANALYSIS:")
print(f"Original length: {len(chunk_0['chunk_text']):,} chars")
print(f"Translated length: {len(chunk_0['translated_text']):,} chars")
print(f"Ratio: {len(chunk_0['translated_text']) / len(chunk_0['chunk_text']):.2f}")

print("\n🔍 VISUAL CHECK:")
print("Does the translated text end mid-sentence? (Truncated)")
print("Or does it end at a natural completion point? (Complete)")

In [0]:
# View parent sections with their AI-generated summaries
print("Parent sections with AI summaries...")
display(
    spark.table("main.default.gdpr_parent_child_chunks")
    .filter(col("child_index") == 0)  # First child of each parent
    .select(
        "source_file_name",
        "parent_id",
        "parent_header",
        "parent_context"  # Header + Summary
    )
    .orderBy("source_file_name", "parent_id")
)

In [0]:
%sql
-- SQL version showing hierarchy
SELECT 
    source_file_name,
    parent_id,
    parent_header,
    parent_context,
    child_index,
    child_type,
    substring(child_text, 1, 200) as original_chunk,
    substring(translated_child, 1, 200) as translated_chunk
FROM main.default.gdpr_parent_child_chunks
ORDER BY source_file_name, parent_id, child_index;

In [0]:
# Test to find the working token limit for ai_translate()
from pyspark.sql.functions import col, expr, lit
import time

print("Testing ai_translate() token limits...")

# Get a sample document
test_df = spark.table("main.default.gdpr_raw_multilingual_precedents").limit(1)
test_text = test_df.select("raw_text_payload").collect()[0]["raw_text_payload"]

print(f"Test document length: {len(test_text)} chars (~{len(test_text)//4} tokens)\n")

# Test progressively larger chunks
test_sizes = [
    ("2K tokens", 8000),
    ("4K tokens", 16000),
    ("6K tokens", 24000),
    ("8K tokens", 32000),
]

print("="*70)
results = []

for label, chars in test_sizes:
    chunk = test_text[:chars]
    print(f"\nTesting {label} ({chars} chars)...")
    
    try:
        start = time.time()
        result = spark.createDataFrame([(chunk,)], ["text"]) \
            .withColumn("translated", expr("ai_translate(text, 'en')")) \
            .select("translated").collect()[0]["translated"]
        
        elapsed = time.time() - start
        
        # Check if translation looks complete (not truncated)
        input_len = len(chunk)
        output_len = len(result) if result else 0
        ratio = output_len / input_len if input_len > 0 else 0
        
        # Normal translation ratio is 0.8-1.3 for most languages
        looks_good = ratio > 0.6 and output_len > 100
        
        status = "✓ SUCCESS" if looks_good else "⚠️ SUSPICIOUS"
        print(f"  {status} - {elapsed:.1f}s")
        print(f"  Input: {input_len:,} chars | Output: {output_len:,} chars | Ratio: {ratio:.2f}")
        print(f"  Preview: {result[:150]}...")
        
        results.append((label, chars, "OK" if looks_good else "TRUNCATED", elapsed, ratio))
        
    except Exception as e:
        print(f"  ✗ FAILED: {str(e)[:100]}")
        results.append((label, chars, f"ERROR: {str(e)[:30]}", 0, 0))
        break

print("\n" + "="*70)
print("RECOMMENDATION:")
print("="*70)

successful = [r for r in results if r[2] == "OK"]
if successful:
    best = max(successful, key=lambda x: x[1])
    print(f"✓ Use {best[0]} ({best[1]:,} chars) per chunk")
    print(f"  Translation ratio: {best[4]:.2f}")
    print(f"  Processing time: {best[3]:.1f}s per chunk")
else:
    print("⚠️ All tests failed or showed truncation. Try 2K tokens max.")

In [0]:
%sql
-- View all translated chunks
SELECT 
    source_file_name,
    chunk_index,
    chunk_tokens,
    substring(chunk_text, 1, 150) as original_preview,
    substring(translated_text, 1, 150) as translated_preview,
    length(chunk_text) as original_length,
    length(translated_text) as translated_length,
    round(length(translated_text) / length(chunk_text), 2) as translation_ratio
FROM main.default.gdpr_translated_chunks
ORDER BY source_file_name, chunk_index

### Generating Embeddings

In [0]:
%pip install sentence-transformers

# ============================================================================
# VECTORIZATION PIPELINE - Automatic Batch Processing with Rate Limiting
# ============================================================================
from pyspark.sql.functions import col, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, FloatType
import pandas as pd
import time
from openai import OpenAI, RateLimitError

print("="*70)
print("AUTOMATIC VECTORIZATION WITH RATE LIMIT HANDLING")
print("="*70)

# ============================================================================
# CONFIGURATION
# ============================================================================
BATCH_SIZE = 500  # Process 500 chunks per batch
TABLE_NAME = "main.default.gdpr_chunk_embeddings"
API_BATCH_SIZE = 30  # Send 30 texts per OpenAI API call (reduced to avoid rate limits)
SLEEP_BETWEEN_BATCHES = 2  # Seconds to wait between API calls

# Get API key from secrets
OPENAI_API_KEY = dbutils.secrets.get(scope="openai", key="GDPR_agent")
client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ API key loaded from secrets")

# ============================================================================
# FUNCTION: Get embeddings with retry logic
# ============================================================================
def get_embeddings_with_retry(texts, batch_size=30, max_retries=5):
    """Get embeddings from OpenAI with automatic retry on rate limits"""
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_texts = [text[:8000] for text in batch_texts]  # Truncate to OpenAI limit
        
        # Retry logic for rate limits
        for attempt in range(max_retries):
            try:
                response = client.embeddings.create(
                    input=batch_texts,
                    model="text-embedding-3-small"
                )
                
                batch_embeddings = [item.embedding for item in response.data]
                embeddings.extend(batch_embeddings)
                
                # Small delay to avoid hitting rate limits
                time.sleep(SLEEP_BETWEEN_BATCHES)
                break  # Success - exit retry loop
                
            except RateLimitError as e:
                if attempt < max_retries - 1:
                    # Extract wait time from error message if available
                    wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s, 8s, 16s
                    print(f"    ⚠️ Rate limit hit. Waiting {wait_time}s (retry {attempt + 1}/{max_retries})...")
                    time.sleep(wait_time)
                else:
                    print(f"    ❌ Failed after {max_retries} retries")
                    raise
        
        # Progress update every 10 API calls
        if (i // batch_size + 1) % 10 == 0:
            print(f"    Processed {i + len(batch_texts)}/{len(texts)} texts...")
    
    return embeddings

# ============================================================================
# MAIN PROCESSING LOOP
# ============================================================================
print("\n" + "="*70)
print("STARTING AUTOMATIC BATCH PROCESSING")
print("="*70)

total_batches_processed = 0
start_time = time.time()

while True:
    # ========================================================================
    # CHECK PROGRESS
    # ========================================================================
    try:
        existing_df = spark.table(TABLE_NAME)
        processed_chunks = existing_df.count()
    except:
        processed_chunks = 0
    
    # Load all chunks
    chunks_df = spark.table("main.default.gdpr_translated_chunks")
    chunks_df = chunks_df.withColumn(
        "chunk_id",
        expr("concat(source_file_name, '_', cast(chunk_index as string))")
    )
    
    total_chunks = chunks_df.count()
    
    # Filter out already processed
    if processed_chunks > 0:
        existing_ids = [row.chunk_id for row in existing_df.select("chunk_id").collect()]
        unprocessed_df = chunks_df.filter(~col("chunk_id").isin(existing_ids))
    else:
        unprocessed_df = chunks_df
    
    remaining = unprocessed_df.count()
    
    # Check if done
    if remaining == 0:
        elapsed = (time.time() - start_time) / 60
        print("\n" + "="*70)
        print("🎉 ALL CHUNKS VECTORIZED!")
        print("="*70)
        print(f"Total chunks processed: {total_chunks:,}")
        print(f"Total batches: {total_batches_processed}")
        print(f"Total time: {elapsed:.1f} minutes")
        print("="*70)
        
        # Show sample
        print("\n📊 Sample embeddings:")
        display(spark.table(TABLE_NAME).select(
            "source_file_name",
            "chunk_index",
            expr("substring(translated_text, 1, 100)").alias("text_preview"),
            expr("size(embedding)").alias("embedding_dimension")
        ).limit(5))
        break
    
    # ========================================================================
    # PROCESS NEXT BATCH
    # ========================================================================
    batch_df = unprocessed_df.limit(BATCH_SIZE)
    batch_count = batch_df.count()
    total_batches_processed += 1
    
    print(f"\n{'─'*70}")
    print(f"BATCH #{total_batches_processed}")
    print(f"{'─'*70}")
    print(f"Processing: {batch_count:,} chunks")
    print(f"Progress: {processed_chunks:,}/{total_chunks:,} ({processed_chunks/total_chunks*100:.1f}%)")
    print(f"Remaining: {remaining:,}")
    
    # Collect batch
    batch_pdf = batch_df.toPandas()
    
    # ========================================================================
    # GENERATE EMBEDDINGS
    # ========================================================================
    print(f"Generating embeddings...")
    batch_start = time.time()
    
    embeddings = get_embeddings_with_retry(
        batch_pdf['translated_text'].tolist(),
        batch_size=API_BATCH_SIZE
    )
    
    batch_elapsed = (time.time() - batch_start) / 60
    print(f"  ✓ Embeddings generated in {batch_elapsed:.1f} minutes")
    
    # Add embeddings to dataframe
    batch_pdf['embedding'] = embeddings
    
    # ========================================================================
    # SAVE BATCH
    # ========================================================================
    print(f"Saving batch...")
    
    schema = StructType([
        StructField("source_file_name", StringType(), True),
        StructField("total_pages", IntegerType(), True),
        StructField("chunk_index", IntegerType(), True),
        StructField("chunk_text", StringType(), True),
        StructField("chunk_tokens", IntegerType(), True),
        StructField("translated_text", StringType(), True),
        StructField("chunk_id", StringType(), True),
        StructField("embedding", ArrayType(FloatType()), True)
    ])
    
    batch_spark_df = spark.createDataFrame(batch_pdf, schema=schema)
    
    if processed_chunks > 0:
        batch_spark_df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)
    else:
        batch_spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)
    
    print(f"  ✓ Batch saved!")
    
    # Show estimated time remaining
    elapsed = (time.time() - start_time) / 60
    chunks_processed_so_far = processed_chunks + batch_count
    rate = chunks_processed_so_far / elapsed if elapsed > 0 else 0
    remaining_after_batch = remaining - batch_count
    
    if rate > 0 and remaining_after_batch > 0:
        est_remaining = remaining_after_batch / rate
        print(f"  ⏱️ Estimated time remaining: {est_remaining:.1f} minutes")

print("\n" + "="*70)
print("VECTORIZATION COMPLETE!")
print("="*70)

### Updating Vector Search Index

In [0]:
# ============================================================================
# INSTALL REQUIRED LIBRARY (run this cell first)
# ============================================================================
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
# ============================================================================
# SYNC VECTOR SEARCH INDEX - GDPR Enforcement Tracker
# ============================================================================
from databricks.vector_search.client import VectorSearchClient

print("="*70)
print("🔄 SYNCING GDPR ENFORCEMENT TRACKER VECTOR INDEX")
print("="*70)

vsc = VectorSearchClient()

try:
    # Get the enforcement tracker index
    index = vsc.get_index(
        endpoint_name="gdpr_rag_endpoint",
        index_name="main.default.gdpr_fines_vector_index"
    )
    
    print("\n📊 Index status before sync:")
    status = index.describe().get('status', {})
    print(f"   State: {status.get('detailed_state', 'UNKNOWN')}")
    print(f"   Ready: {'✅' if status.get('ready', False) else '❌'}")
    
    # Trigger sync
    print("\n🔄 Triggering sync...")
    index.sync()
    print("   ✅ Sync triggered successfully")
    
    print("\n" + "="*70)
    print("✅ ENFORCEMENT TRACKER INDEX SYNC COMPLETE")
    print("="*70)
    print("\nℹ️  Note: Sync is asynchronous and may take 2-5 minutes to complete.")
    print("   The index will reflect new enforcement tracker data once sync finishes.")
    
except Exception as e:
    print(f"\n❌ Error syncing index: {e}")
    print("\nℹ️  This might happen if:")
    print("   • Embeddings table doesn't exist yet")
    print("   • Index hasn't been created (run vector search setup first)")
    print("   • Endpoint is offline")